# HHS Unaccompanied Children Program: Exploratory Data Analysis

This notebook reproduces the descriptive analysis in `RESEARCH_PAPER.md`. It is designed to run in Google Colab or locally. Upload the CSV when the repository data file is unavailable. The data contains periodic operational snapshots, not a complete daily time series; totals are not unique-child counts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind')
DATA_PATH = Path('data/HHS_Unaccompanied_Alien_Children_Program.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        df = pd.read_csv(uploaded_name)
    except ImportError as exc:
        raise FileNotFoundError('Place the CSV at data/HHS_Unaccompanied_Alien_Children_Program.csv or upload it in Colab.') from exc

df.head()

In [ ]:
DATE = 'Date'
MEASURES = {
    'Apprehended': 'Children apprehended and placed in CBP custody*',
    'In CBP custody': 'Children in CBP custody',
    'Transferred out': 'Children transferred out of CBP custody',
    'In HHS care': 'Children in HHS Care',
    'Discharged': 'Children discharged from HHS Care',
}
required = {DATE, *MEASURES.values()}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')
df[DATE] = pd.to_datetime(df[DATE], errors='raise')
for column in MEASURES.values():
    df[column] = pd.to_numeric(df[column].astype(str).str.replace(',', '', regex=False), errors='raise')
df = df.sort_values(DATE).reset_index(drop=True)
df.info()

In [ ]:
summary = df[list(MEASURES.values())].describe().T
summary.index = MEASURES.keys()
summary[['count', 'mean', '50%', 'min', 'max']].rename(columns={'50%': 'median'}).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df[DATE], df[MEASURES['In HHS care']], color='#2563eb', linewidth=2)
ax.set_title('Children in HHS care over reported observations')
ax.set_xlabel('Observation date')
ax.set_ylabel('Children')
fig.autofmt_xdate()
plt.show()

In [ ]:
flow_names = ['Apprehended', 'In CBP custody', 'Transferred out', 'Discharged']
flow_frame = df[[DATE] + [MEASURES[name] for name in flow_names]].rename(columns={MEASURES[name]: name for name in flow_names})
fig, ax = plt.subplots(figsize=(13, 6))
for name in flow_names:
    ax.plot(flow_frame[DATE], flow_frame[name], label=name, linewidth=1.5)
ax.set_title('Operational flows')
ax.set_xlabel('Observation date')
ax.set_ylabel('Reported children')
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
long = df.melt(id_vars=DATE, value_vars=list(MEASURES.values()), var_name='Measure', value_name='Children')
long['Measure'] = long['Measure'].map({value: key for key, value in MEASURES.items()})
plt.figure(figsize=(12, 6))
sns.boxplot(data=long, x='Measure', y='Children')
plt.title('Distribution of reported measures')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
care = MEASURES['In HHS care']
peak = df.loc[df[care].idxmax()]
latest = df.iloc[-1]
print(f"Rows: {len(df):,}")
print(f"Date range: {df[DATE].min():%B %d, %Y} to {df[DATE].max():%B %d, %Y}")
print(f"Peak HHS care: {peak[care]:,.0f} on {peak[DATE]:%B %d, %Y}")
print(f"Latest HHS care: {latest[care]:,.0f} on {latest[DATE]:%B %d, %Y}")
print(f"Latest versus peak: {(latest[care] / peak[care] - 1) * 100:.1f}%")
print(f"Duplicate dates: {df[DATE].duplicated().sum():,}")
print('Note: missing dates are not zero activity, and sums are not unique-child totals.')

## Key insights

The following cells calculate the findings directly from the loaded data rather than relying on hard-coded paper values. These are descriptive observations, not causal conclusions.

1. **Program scale changed materially.** Compare the maximum reported HHS-care population with the final observation.
2. **Demand is uneven.** The range and percentile statistics show why capacity planning should not rely on the mean alone.
3. **February 2024 merits operational review.** Peak observations across apprehension, CBP custody, and transfers should be reviewed alongside policy, weather, border, facility, and case-management context.
4. **Late-2025 demand remains non-zero.** A lower endpoint does not mean that operational capacity is no longer needed.
5. **The reporting cadence is irregular.** Missing calendar dates are missing observations, not evidence of zero activity.


In [ ]:
care = MEASURES['In HHS care']
peak = df.loc[df[care].idxmax()]
latest = df.iloc[-1]
flow_peak_rows = {
    name: df.loc[df[MEASURES[name]].idxmax()]
    for name in ['Apprehended', 'In CBP custody', 'Transferred out', 'Discharged']
}
insights = pd.DataFrame({
    'Metric': ['Observations', 'Calendar-day span', 'Latest HHS care', 'Peak HHS care', 'Latest versus peak'],
    'Value': [
        f"{len(df):,}",
        f"{(df[DATE].max() - df[DATE].min()).days + 1:,}",
        f"{latest[care]:,.0f} ({latest[DATE]:%B %d, %Y})",
        f"{peak[care]:,.0f} ({peak[DATE]:%B %d, %Y})",
        f"{(latest[care] / peak[care] - 1) * 100:.1f}%",
    ],
})
display(insights)
for name, row in flow_peak_rows.items():
    print(f"{name} peak: {row[MEASURES[name]]:,.0f} on {row[DATE]:%B %d, %Y}")
print(f"Transfers out minus CBP custody totals: {df[MEASURES['Transferred out']].sum() - df[MEASURES['In CBP custody']].sum():,.0f}")
print('This difference is not a backlog measure because the fields may not describe aligned cohorts or time periods.')

In [ ]:
percentiles = df[list(MEASURES.values())].quantile([0.5, 0.9]).T
percentiles.index = MEASURES.keys()
percentiles.columns = ['Median', '90th percentile']
display(percentiles.round(1))
print('Planning implication: use rolling medians and high-percentile thresholds instead of the full-period mean alone.')

## Recommendations for government and program stakeholders

### Immediate: 0–90 days

- Establish a weekly operational review using HHS care, apprehensions, transfers, and discharges together.
- Define an exception threshold using a rolling 30-observation median and 90th percentile; escalate when multiple measures are elevated together.
- Add automated checks for duplicate dates, missing dates, unexpected negative values, and schema changes.
- Use the observed HHS-care range to maintain both baseline capacity and surge capacity.

### Near term: 3–12 months

- Publish a machine-readable data dictionary, release identifier, reporting cadence, and revision history.
- Add capacity, occupied beds, pending placements, length of stay, and discharge pathway fields where privacy and policy permit.
- Reconcile the timing definitions for custody and transfer measures before presenting them as a flow balance.
- Build forecasts with prediction intervals and explicit missing-observation handling.

### Longer term

- Evaluate operational interventions with pre-specified metrics rather than simple before/after comparisons.
- Create privacy-preserving geographic and demographic aggregates for equity monitoring.
- Independently validate data lineage from source systems to public release.

## Limitations and responsible interpretation

- This is exploratory descriptive analysis; it does not estimate causal effects, forecast demand, evaluate program quality, or represent all children served.
- Totals are sums of reported observations, not unique-child counts.
- A missing date means no observation in the extract, not necessarily zero activity.
- Aggregate data cannot support individual-level conclusions or explain why peaks occurred.
- The extract lacks facility, state, age, sex, referral, discharge destination, length-of-stay, and denominator fields.
- The source may revise the extract, so decisions should reference the release date and file version.
- Protect children’s privacy by avoiding personally identifiable information and small-cell disclosure.
